# Section 4 - Workflows, Part 2: Handoff & Orchestration

## Storyline

Section 3 was deterministic — we knew exactly which steps would run. Real customer calls aren't like that. A representative starts with PolicyPal, and depending on what the customer says, the conversation needs to **branch**:

- *"I want to file a claim"* → hand off to **ClaimsAgent**
- *"I want to change my contribution or product risk profile"* → hand off to **RiskAgent**
- *"I'm worried about market volatility"* → hand off to **RiskAgent**

PolicyPal becomes a **triage / dispatch** agent. The specialists own the actual work. This is the **handoff** pattern.

## What you'll build

| Pattern | Use-case | When to choose it |
|---------|------------------|-------------------|
| **Handoff** | PolicyPal triages, then transfers control to the right specialist | Mostly-linear flows with a decision point |
| **Magentic** (mention) | A senior rep asks: *"Build me a 5-year retirement scenario for this customer"* — open-ended, multi-step | Open-ended tasks where the *plan* itself must be discovered |

We'll **build** handoff and **discuss** Magentic; full samples live in [`microsoft/agent-framework — python/samples/04-workflows-orchestrations`](https://github.com/microsoft/agent-framework/tree/main/python/samples/04-workflows-orchestrations).

## Setup — specialist tools

Each specialist gets a small focused tool that simulates a real Athora Netherlands back-office call.

In [ ]:
import os
from typing import Annotated, cast
from agent_framework import Agent, AgentResponse, WorkflowEvent, WorkflowRunState, tool
from agent_framework.orchestrations import HandoffBuilder, HandoffAgentUserRequest
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

@tool(approval_mode="never_require")
def open_claim(policy_number: Annotated[str, "Policy number"], description: Annotated[str, "Short description"]) -> str:
    """Open a new Athora Netherlands claim file. Returns the claim ID."""
    return f"Claim opened: ATH-CLM-{abs(hash(policy_number)) % 100000:05d} — '{description}'"

@tool(approval_mode="never_require")
def review_product_change(policy_number: Annotated[str, "Policy number"], requested_change: Annotated[str, "Requested change"]) -> str:
    """Return risk and compliance guardrails for a requested product or contribution change."""
    return (
        f"Risk review for {policy_number}: {requested_change}. "
        "Confirm suitability, risk profile, and DNB / AFM disclosure requirements before execution."
    )

@tool(approval_mode="never_require")
def explain_risk_profile(policy_number: Annotated[str, "Policy number"]) -> str:
    """Return a plain-language summary of the policy's risk profile."""
    return (
        f"Policy {policy_number} risk profile: balanced (60% equities / 40% bonds). "
        "Recent volatility is within the expected range for this profile."
    )

## Handoff workflow vocabulary

`HandoffBuilder` automatically registers a `transfer_to_<name>` tool on the **coordinator** for every participant. The coordinator chooses whom (if anyone) to transfer to. The chosen specialist then drives the conversation.

In [ ]:
policypal_triage = Agent(
    client=client,
    name="policypal_triage",
    instructions=(
        "You are PolicyPal, Athora Netherlands' frontline assistant. Greet the rep briefly and route the request "
        "to the right specialist by transferring control. Use claims_agent for new claims, "
        "and risk_agent for product changes, contribution changes, or risk-profile questions. "
        "Do not handle these yourself."
    ),
    require_per_service_call_history_persistence=True,
)

claims_agent = Agent(
    client=client,
    name="claims_agent",
    instructions="You handle Athora Netherlands claim intake. Use open_claim once you have a policy number and description.",
    tools=[open_claim],
    require_per_service_call_history_persistence=True,
)

risk_agent = Agent(
    client=client,
    name="risk_agent",
    instructions=(
        "You handle product-change and risk-profile questions. Use review_product_change for requested changes; "
        "use explain_risk_profile for risk explanations. Keep DNB / AFM guardrails visible."
    ),
    tools=[review_product_change, explain_risk_profile],
    require_per_service_call_history_persistence=True,
)

handoff_workflow = (
    HandoffBuilder(
        participants=[policypal_triage, claims_agent, risk_agent],
        termination_condition=lambda conversation: (
            len(conversation) > 0 and "that's all" in conversation[-1].text.lower()
        ),
    )
    .with_start_agent(policypal_triage)
    .build()
)
print("Handoff workflow built.")

## Execute one scripted handoff

Handoff workflows are **conversational**: they pause to ask the user (the rep) for input, run a turn, and pause again. We'll script the rep's side here for reproducibility.

In [ ]:
# Scripted rep responses for reproducibility
scripted_responses = [
    "A customer wants to lower their pension contribution on policy NL-2031-887 to 300 EUR/month.",
    "Yes, please proceed.",
    "Thanks, that's all.",
]

def handle_events(events: list[WorkflowEvent]) -> list[WorkflowEvent]:
    """Process events: print outputs + collect pending user-input requests."""
    requests = []
    for event in events:
        if event.type == "handoff_sent":
            print(f"\n[Handoff: {event.data.source} → {event.data.target}]")
        elif event.type == "output" and isinstance(event.data, AgentResponse):
            for msg in event.data.messages:
                if msg.text:
                    speaker = msg.author_name or msg.role
                    print(f"  {speaker}: {msg.text}")
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            requests.append(event)
        elif event.type == "status":
            print(f"\n[Status: {event.state}]")
    return requests

# Start the workflow
print("Starting handoff workflow...\n")
initial_message = scripted_responses.pop(0)
print(f"  rep: {initial_message}")
stream = handoff_workflow.run(initial_message, stream=True)
pending = handle_events([event async for event in stream])

# Request/response loop
while pending:
    if not scripted_responses:
        responses = {req.request_id: HandoffAgentUserRequest.terminate() for req in pending}
    else:
        user_msg = scripted_responses.pop(0)
        print(f"\n  rep: {user_msg}")
        responses = {req.request_id: HandoffAgentUserRequest.create_response(user_msg) for req in pending}

    result = await handoff_workflow.run(responses=responses)
    pending = handle_events(result)

print("\n✓ Handoff workflow complete.")

### What just happened

- `policypal_triage` was the only agent that started talking. It saw the rep's first message and decided to call `transfer_to_pension_change_agent` (auto-generated by `HandoffBuilder`).
- `risk_agent` took over, checked suitability and regulatory guardrails, then used its specialist tool.
- The workflow stayed in a request/response loop with the rep until it ran out of work.

**Important:** the specialist agents never see each other. The coordinator never re-enters once it has handed off (in this `handoff_simple` shape). For richer flows where control bounces back, see `handoff_autonomous.py` in the framework samples.

## A note on Magentic (open-ended orchestration)

Sequential / Concurrent / Handoff are all patterns where *we* defined the structure. Some of your tasks don't have a known structure ahead of time:

> *"For this customer: pull policy NL-2031-887, project 5, 10, and 20 years out, compare to their stated retirement goal, and tell me whether to recommend a contribution increase."*

**Magentic** orchestration discovers a plan, executes steps, optionally re-plans, and can pause for a human review (`magentic_human_plan_review.py`). It is the closest pattern to *autonomous agents* in Agent Framework.



## Handoff design notes

Choose handoff when the route depends on what the rep or customer says at run time. Choose sequential when the route is fixed by design. For insurance companies, handoff is useful for claims vs. risk vs. complaints, but regulated product changes should still end with an explicit compliance checkpoint.

## Chapter 2 Complete! 🎉

- Handoff = coordinator + specialists + auto-registered transfer tools.
- The coordinator decides; specialists execute; the workflow handles request/response with the user.
- Magentic is the framework's answer for *open-ended* tasks; covered as further reading.

**Coming up in Next Chapter:** all of this is great when it works. When PolicyPal hands off to the wrong specialist, or `change_contribution` is called with the wrong number, you need **traces**. We'll instrument everything you've built today and see it in Foundry tracing + Application Insights.